# Basket Affinity Analysis

## Project Overview
Analyze transaction baskets to discover which products are frequently bought together. This project uses association rule mining (Apriori) to find product affinities and derives merchandising recommendations from the results.

## Learning Objectives
- Understand market basket analysis and its business applications
- Transform transaction data into basket format
- Apply the Apriori algorithm to find frequent itemsets
- Interpret support, confidence, and lift metrics
- Generate actionable cross-selling and merchandising insights

## Problem Statement
A retailer wants to understand which products customers tend to buy together so they can optimize product placement, create bundle promotions, and improve cross-sell recommendations.

## Why This Project Matters
Market basket analysis is one of the most commercially impactful data analysis techniques. Amazon's "Customers who bought this also bought..." feature is built on this principle. Even small retailers can benefit from understanding product affinities.

## Dataset Overview
We generate synthetic grocery transaction data with realistic product co-purchase patterns. Each transaction represents a customer's basket of items.

## Dataset Source and License Notes
- Synthetic data generated for educational purposes
- Patterns inspired by common grocery shopping behavior
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from collections import Counter

sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
MIN_SUPPORT = 0.02  # minimum support threshold
MIN_CONFIDENCE = 0.15  # minimum confidence threshold
MIN_LIFT = 1.2  # minimum lift for interesting rules
print('Configuration set')

## Dataset Download or Loading
We generate synthetic transaction data with built-in product affinities (e.g., bread+butter, chips+salsa).

In [ ]:
np.random.seed(42)
n_transactions = 5000

products = ['Bread', 'Butter', 'Milk', 'Eggs', 'Cheese', 'Chips', 'Salsa',
            'Beer', 'Diapers', 'Coffee', 'Sugar', 'Cereal', 'Bananas',
            'Apples', 'Yogurt', 'Chicken', 'Rice', 'Pasta', 'Tomato Sauce',
            'Onions', 'Garlic', 'Olive Oil', 'Wine', 'Chocolate', 'Ice Cream']

# Create baskets with built-in affinities
baskets = []
for i in range(n_transactions):
    basket = set()
    # Random base items
    n_items = np.random.randint(2, 8)
    basket.update(np.random.choice(products, size=min(n_items, len(products)), replace=False))

    # Inject co-purchase patterns
    if 'Bread' in basket and np.random.random() < 0.65:
        basket.add('Butter')
    if 'Chips' in basket and np.random.random() < 0.55:
        basket.add('Salsa')
    if 'Pasta' in basket and np.random.random() < 0.60:
        basket.add('Tomato Sauce')
    if 'Beer' in basket and np.random.random() < 0.40:
        basket.add('Diapers')  # classic association
    if 'Coffee' in basket and np.random.random() < 0.50:
        basket.add('Sugar')
    if 'Chicken' in basket and np.random.random() < 0.45:
        basket.add('Rice')
    if 'Onions' in basket and np.random.random() < 0.50:
        basket.add('Garlic')

    baskets.append(list(basket))

# Create transaction DataFrame
rows = []
for tid, basket in enumerate(baskets):
    for item in basket:
        rows.append({'TransactionID': tid, 'Product': item})

df = pd.DataFrame(rows)
print(f'Transactions: {len(baskets)}, Total rows: {len(df)}')
print(f'Unique products: {df["Product"].nunique()}')
df.head(10)

## Data Validation Checks

In [ ]:
print(f'Missing values:\n{df.isnull().sum()}')
print(f'\nItems per basket:')
basket_sizes = df.groupby('TransactionID')['Product'].count()
print(basket_sizes.describe().round(2))
print(f'\nSingle-item baskets: {(basket_sizes == 1).sum()}')
print(f'Empty baskets: {(basket_sizes == 0).sum()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Product frequency
product_freq = df['Product'].value_counts()
product_freq.plot.barh(ax=axes[0], edgecolor='black')
axes[0].set_title('Product Frequency Across All Baskets')
axes[0].set_xlabel('Occurrences')
axes[0].invert_yaxis()

# Basket size distribution
basket_sizes.plot.hist(ax=axes[1], bins=range(1, 15), edgecolor='black', alpha=0.7)
axes[1].set_title('Basket Size Distribution')
axes[1].set_xlabel('Number of Items')
axes[1].set_ylabel('Number of Baskets')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_products.png'), dpi=100, bbox_inches='tight')
plt.show()

## Target Analysis
In basket analysis there's no prediction target. The "target" is discovering frequently co-purchased product pairs and their association strength.

In [ ]:
# Top co-occurring pairs
pair_counts = Counter()
for basket in baskets:
    for pair in combinations(sorted(basket), 2):
        pair_counts[pair] += 1

top_pairs = pd.DataFrame(pair_counts.most_common(20), columns=['Pair', 'Count'])
top_pairs['Support'] = top_pairs['Count'] / len(baskets)
print('Top 20 Co-Occurring Product Pairs:')
print(top_pairs.to_string(index=False))

## Association Rule Mining (Manual Implementation)
We compute support, confidence, and lift for product pair rules without requiring external libraries. This makes the logic transparent and educational.

In [ ]:
# Build binary basket matrix
basket_sets = [set(b) for b in baskets]
n = len(basket_sets)

# Item support
item_support = {}
for prod in products:
    count = sum(1 for b in basket_sets if prod in b)
    item_support[prod] = count / n

# Pair support, confidence, lift
rules = []
for a, b in combinations(products, 2):
    both = sum(1 for bskt in basket_sets if a in bskt and b in bskt)
    supp = both / n
    if supp < MIN_SUPPORT:
        continue

    # Rule: A -> B
    conf_ab = both / sum(1 for bskt in basket_sets if a in bskt) if item_support[a] > 0 else 0
    lift_ab = conf_ab / item_support[b] if item_support[b] > 0 else 0

    # Rule: B -> A
    conf_ba = both / sum(1 for bskt in basket_sets if b in bskt) if item_support[b] > 0 else 0
    lift_ba = conf_ba / item_support[a] if item_support[a] > 0 else 0

    rules.append({'Antecedent': a, 'Consequent': b, 'Support': round(supp, 4),
                  'Confidence': round(conf_ab, 4), 'Lift': round(lift_ab, 4)})
    rules.append({'Antecedent': b, 'Consequent': a, 'Support': round(supp, 4),
                  'Confidence': round(conf_ba, 4), 'Lift': round(lift_ba, 4)})

rules_df = pd.DataFrame(rules)
print(f'Total rules (support >= {MIN_SUPPORT}): {len(rules_df)}')

## Filtering High-Quality Rules

In [ ]:
strong_rules = rules_df[
    (rules_df['Confidence'] >= MIN_CONFIDENCE) &
    (rules_df['Lift'] >= MIN_LIFT)
].sort_values('Lift', ascending=False).reset_index(drop=True)

print(f'Strong rules (conf >= {MIN_CONFIDENCE}, lift >= {MIN_LIFT}): {len(strong_rules)}')
print()
print(strong_rules.head(20).to_string(index=False))

## Visualizing Top Rules

In [ ]:
top_rules = strong_rules.head(15).copy()
top_rules['Rule'] = top_rules['Antecedent'] + ' → ' + top_rules['Consequent']

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Lift chart
axes[0].barh(top_rules['Rule'], top_rules['Lift'], edgecolor='black', color='steelblue')
axes[0].axvline(x=1.0, color='red', linestyle='--', label='Lift = 1 (random)')
axes[0].set_title('Top 15 Rules by Lift')
axes[0].set_xlabel('Lift')
axes[0].legend()
axes[0].invert_yaxis()

# Confidence chart
axes[1].barh(top_rules['Rule'], top_rules['Confidence'], edgecolor='black', color='coral')
axes[1].set_title('Top 15 Rules by Confidence')
axes[1].set_xlabel('Confidence')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'top_rules.png'), dpi=100, bbox_inches='tight')
plt.show()

## Support vs Confidence Scatter

In [ ]:
plt.figure(figsize=(10, 7))
scatter = plt.scatter(rules_df['Support'], rules_df['Confidence'],
                      c=rules_df['Lift'], cmap='RdYlGn', s=40, alpha=0.7, edgecolors='grey')
plt.colorbar(scatter, label='Lift')
plt.axhline(y=MIN_CONFIDENCE, color='red', linestyle='--', alpha=0.5, label=f'Min Confidence ({MIN_CONFIDENCE})')
plt.axvline(x=MIN_SUPPORT, color='blue', linestyle='--', alpha=0.5, label=f'Min Support ({MIN_SUPPORT})')
plt.xlabel('Support')
plt.ylabel('Confidence')
plt.title('Association Rules: Support vs Confidence (colored by Lift)')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'support_confidence.png'), dpi=100, bbox_inches='tight')
plt.show()

## Product Affinity Heatmap

In [ ]:
# Build product co-occurrence matrix
top_products = product_freq.head(12).index.tolist()
cooccur = pd.DataFrame(0, index=top_products, columns=top_products)
for bskt in basket_sets:
    items_in = [p for p in top_products if p in bskt]
    for a, b in combinations(items_in, 2):
        cooccur.loc[a, b] += 1
        cooccur.loc[b, a] += 1

plt.figure(figsize=(10, 8))
sns.heatmap(cooccur, annot=True, fmt='d', cmap='YlOrRd')
plt.title('Product Co-Occurrence Matrix (Top 12 Products)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'cooccurrence_heatmap.png'), dpi=100, bbox_inches='tight')
plt.show()

## Merchandising Recommendations
Based on the association rules with the highest lift and confidence:

In [ ]:
print('=' * 60)
print('MERCHANDISING RECOMMENDATIONS')
print('=' * 60)
recs = strong_rules.head(10)
for _, row in recs.iterrows():
    print(f"\n  If customer buys '{row['Antecedent']}',")
    print(f"    suggest '{row['Consequent']}'")
    print(f"    Confidence: {row['Confidence']:.1%} | Lift: {row['Lift']:.2f}")
print()
print('Actions:')
print('  1. Place high-lift pairs adjacent on shelves')
print('  2. Create bundle discounts for top confident pairs')
print('  3. Use top rules for online "frequently bought together" widgets')
print('  4. Design cross-aisle promotional signage for strong pairs')

## Interpretation and Business Insight
- **Bread → Butter** and **Pasta → Tomato Sauce** show intuitive meal-prep patterns with high lift
- **Beer → Diapers** appears as a real-world classic (often attributed to convenience-store shopping patterns)
- Rules with lift > 1.5 represent genuinely strong affinities, not just popular items co-appearing by chance
- High-confidence, low-support rules may represent niche but valuable cross-sell opportunities

## Limitations
- Synthetic data with injected patterns — real data would have subtler, noisier associations
- Pair-level analysis only (no 3+ item rules) for simplicity
- No temporal dimension: purchase affinities may vary by season or time of day
- Support/confidence thresholds are manually set; optimal thresholds depend on business context

## How to Improve This Project
- Use real POS data for more realistic pattern discovery
- Implement the Apriori algorithm for 3+ item rules using `mlxtend`
- Add temporal basket analysis: do affinities change by day of week or season?
- Incorporate product category hierarchy for higher-level affinities
- Apply network graph visualization for product relationship mapping

## Production Considerations
- Run association mining on a weekly or monthly cadence
- Filter rules by actionability: can the business actually act on this pair?
- A/B test bundle promotions derived from top rules
- Monitor rule stability over time — volatile rules may not be reliable

## Common Mistakes
- Including extremely popular items without normalizing (they co-occur with everything)
- Ignoring lift and relying only on confidence (misleading for popular items)
- Using random splits instead of keeping baskets intact
- Not removing returns or cancelled transactions before analysis

## Mini Challenge / Exercises
1. Change MIN_SUPPORT to 0.01. How many more rules appear? Are they useful?
2. Add a 3rd product to each rule (3-item frequent itemsets). What patterns emerge?
3. Filter to baskets with 5+ items only. Do the top rules change?
4. Create a simple recommendation function: given a product, return the top 3 suggested products.

## Final Summary / Key Takeaways
- Market basket analysis reveals hidden product affinities in transaction data
- Support measures frequency, confidence measures reliability, lift measures true association strength
- Lift > 1 means a genuine positive association beyond random chance
- Actionable insights include shelf placement, bundling, and cross-sell recommendations
- Even a simple pair-based analysis provides significant business value